# 05 · Preprocessing pipeline (incl. PCA dimensionality)

Runnable, documented front-end to `scripts/preprocessing/run_preprocessing.py`, so the data
workflow is easy to follow and reproduce.

The pipeline has **five ordered steps** (`STEP_ORDER` in `run_preprocessing.py`):

| # | step | what it does | output |
|---|------|--------------|--------|
| 1 | `convert` | SCP542 CPM + metadata → AnnData, **HVG filter** (`hvg5000` keeps top-5,000 by dispersion on a log1p copy; `.X` stays CPM) | `SCP542_CCLE.h5ad` |
| 2 | `scgpt`   | scGPT_human cell embeddings (separate scGPT venv); **drops scGPT-OOV genes from `.X`** | `..._scGPT_human_embeddings.h5ad` |
| 3 | `targets` | attach CTRPv2 `Y_ctrp` / `M_ctrp` drug-response targets (all 545 drugs) | `..._with_targets.h5ad` |
| 4 | `splits`  | **cell-line-grouped** 70/15/15 split `split_ctrp` (+ `split_paclitaxel`) — leakage-free | (in-place) |
| 5 | `pca`     | **PCA baseline `X_pca`** on the HVG-filtered *convert* counts (not the OOV-dropped `.X`) | (in-place) |

**Key decisions baked in here:**
- **PCA width = 512** (`add_pca.DEFAULT_N_COMPS`, override `--pca-n-comps`) so `X_pca` matches the
  scGPT embedding width — the PCA-vs-scGPT comparison then differs only in *how* genes are encoded.
- **PCA is computed on the convert counts** (full HVG set: 5,000 / 22,722 genes), *not* the targets
  `.X` (which lost scGPT-OOV genes). Keeps PCA a clean single-cell baseline on the single HVG filter.
- Splits are **grouped by cell line**, not by cell — viability is defined per (cell line × drug), so a
  random per-cell split would leak a line across train/val/test.

In [1]:
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.preprocessing.layout import PipelinePaths
from scripts.preprocessing import add_pca

# --- knobs -----------------------------------------------------------------
VARIANTS    = ['hvg5000', 'all_genes']   # both gene-set variants
PCA_N_COMPS = 512                          # PCA baseline width (matches scGPT)
# ---------------------------------------------------------------------------
print('add_pca default n_comps:', add_pca.DEFAULT_N_COMPS)
for v in VARIANTS:
    p = PipelinePaths.build(None, v)
    print(f'{v:9s} -> {p.targets_h5ad}')

add_pca default n_comps: 512
hvg5000   -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad
all_genes -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad


## Full pipeline (reference)

Steps 1–4 are expensive and step 2 (`scgpt`) needs a **separate scGPT venv**, so we don't re-run
them here. Full run per variant:

```bash
uv run scripts/preprocessing/run_preprocessing.py \
    --variant <hvg5000|all_genes> --all-drugs \
    --scgpt-python /path/to/scgpt-venv/bin/python --pca-n-comps 512
```

Below we (re)run **only the PCA step** — the part the 512-d switch changed — for **both variants**
against the already-built `..._with_targets.h5ad`. Idempotent (`--start-at pca --force-pca`).

In [2]:
for variant in VARIANTS:
    cmd = [
        'uv', 'run', 'scripts/preprocessing/run_preprocessing.py',
        '--variant', variant,
        '--start-at', 'pca', '--skip-scgpt', '--force-pca',
        '--pca-n-comps', str(PCA_N_COMPS),
    ]
    print('\n' + '=' * 70 + f'\n{variant}: ' + ' '.join(cmd) + '\n' + '=' * 70)
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print('STDERR:\n', proc.stderr[-2000:])
        raise RuntimeError(f'{variant} preprocessing failed (exit {proc.returncode})')


hvg5000: uv run scripts/preprocessing/run_preprocessing.py --variant hvg5000 --start-at pca --skip-scgpt --force-pca --pca-n-comps 512


data_root : /Users/selin/Desktop/OncoTox/data
variant   : hvg5000 -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000

[5/5] add_pca (force=True, n_comps=512)
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad...
Computing PCA on HVG-filtered counts from /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE.h5ad...
  X_pca computed on 5000 genes -> shape (53513, 512).
Saving updated AnnData with X_pca (targets .X unchanged)...
Done! You can now run baseline training.

Preprocessing pipeline complete.
Final file: /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad


all_genes: uv run scripts/preprocessing/run_preprocessing.py --variant all_genes --start-at pca --skip-scgpt --force-pca --pca-n-comps 512


data_root : /Users/selin/Desktop/OncoTox/data
variant   : all_genes -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes

[5/5] add_pca (force=True, n_comps=512)
Loading /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad...
Computing PCA on HVG-filtered counts from /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE.h5ad...
  X_pca computed on 22722 genes -> shape (53513, 512).
Saving updated AnnData with X_pca (targets .X unchanged)...
Done! You can now run baseline training.

Preprocessing pipeline complete.
Final file: /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE_scGPT_human_embeddings_with_targets.h5ad



## Verify the PCA baseline (both variants)

In [3]:
import scanpy as sc

for variant in VARIANTS:
    p = PipelinePaths.build(None, variant)
    a = sc.read_h5ad(p.targets_h5ad)
    print(f'{variant:9s} | cells x genes {a.shape} | X_pca {a.obsm["X_pca"].shape} '
          f'| X_scGPT {a.obsm["X_scGPT"].shape} '
          f'| split_ctrp {"yes" if "split_ctrp" in a.obs else "NO"} '
          f'| K={len(a.uns.get("ctrp_drugs", []))}')
    assert a.obsm['X_pca'].shape[1] == PCA_N_COMPS
print(f'\nOK — X_pca is {PCA_N_COMPS}-d for both variants. Ready for 07_training.ipynb.')

hvg5000   | cells x genes (53513, 4576) | X_pca (53513, 512) | X_scGPT (53513, 512) | split_ctrp yes | K=545


all_genes | cells x genes (53513, 20570) | X_pca (53513, 512) | X_scGPT (53513, 512) | split_ctrp yes | K=545

OK — X_pca is 512-d for both variants. Ready for 07_training.ipynb.
